<a href="https://colab.research.google.com/github/redinbluesky/handson-llm/blob/main/04_텍스트_분류.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  목차
* [Chapter 4 서론](#chapter4)
* [Chapter 4-1 영화 리뷰 데이터셋](#chapter4-1)
* [Chapter 4-2 표현 모델로 텍스트 분류하기](#chapter4-2)
* [Chapter 4-3 모델 선택](#chapter4-3)
* [Chapter 4-4 작업에 특화된 모델 사용하기](#chapter4-4)
* [Chapter 4-5 임베딩을 활용하여 분류 작업 수행하기](#chapter4-5)
    * [Chapter 4-5-1 지도 학습 분류](#chapter4-5-1)
    * [Chapter 4-5-2 데이터에 레이블이 없는 경우](#chapter4-5-2)
* [Chapter 4-6 생성 모델로 텍스트 분류하기](#chapter4-6)    

## Chapter 4 서론 <a class="anchor" id="chapter4"></a>
1. 가장 일반적인 자연어 처리 작업은 분류이다.
    - 작업의 목표는 입력 텍스트에 레이블 혹은 클래스를 할당하는 것이 목표이다.
    - 예를 들어, 스팸 이메일 분류, 감정 분석, 주제 분류 등이 있다.
    
        ![분류 예시](./image/04_example.png)

2. 이 장에서 언어 모델을 텍스트 분류에 사용하는 몇 가지 방법을 알아본다.
    - 사전 훈련된 모델을 사용하는 방법을 소개한다.
    - 작업에 특화된 모델과 임베딩 모델을 다루어본다.

3. 대규모 데이터에서 이미 훈련되어 텍스트 분류에 활용할 수 있는 사전 훈련된 언어모델을 활용하는데 초점을 맞춘다.

4. 표현 언어 모델과 생성 언어 모델을 살펴보고 둘의 차이점을 알아본다.

    ![언어 모델 비교](./image/04_language_model_comparison.png)

## Chapter 4-1 영화 리뷰 데이터셋 <a class="anchor" id="chapter4-1"></a>
1. 허깅 페이스 허브에는 모델은 물론 텍스트 분류을 위한 데이터도 찾을 수 있다.
    - 유명한 로튼 토마토 데이터셋을 사용하여 모델을 훈련하고 평가한다.
    - 로트 토마토에서 수집한 양성 영화리뷰 5,331개, 음성 영화 리뷰 5,331개가 들어있다.

In [4]:
# datasets 패키지를 사용하여 로튼 토마토 데이터 셋을 로드한다.
from datasets import load_dataset
data = load_dataset("rotten_tomatoes")
data

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

2. 데이터는 훈련, 검증, 테스트 세트로 나누어져 있다.
    - 모델 훈련을 위해 훈련 세트를 사용하고, 결과 확인을 위해 테스트 세트를 사용한다.
    - 모델 훈련 시 하이퍼파라미터를 튜닝하고 싶다면 검증 세트를 사용할 수 있다.

In [10]:
# 훈련 세트 샘플 확인
data["train"][0]

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
 'label': 1}

In [ ]:
# 훈련 세트 샘플 확인
#   - 첫 번째 샘풀과 마자막 샘플을 확인한다.
data["train"][0,-1]

{'text': ['the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
  'things really get weird , though not particularly scary : the movie is all portent and no content .'],
 'label': [1, 0]}

3. 리뷰에는 양성(0)과 음성(1) 레이블이 부여된 것으로보아 이진 분류 문제임을 알 수 있다

## Chapter 4-2 표현 모델로 텍스트 분류하기 <a class="anchor" id="chapter4-2"></a>
1. 사전 훈련된 표현 모델을 사용해 분류를 할때 일반적으로 작업에 특화된 모델이나 임베딩 모델을 사용한다.

2. 이런 모델은 아래의 그림과 같이 BERT와 같은 파운데이션 모델을 특정 후속 작업에 맞춰 미세 튜닝하여 만든다.

    ![표현 모델](./image/04_representation_model.png)

3. 이 장에서는 아래의 그림과 같이 두 모델을 동결하고(훈련하지 않고) 모델의 출력만 사용한다.
    - 이미 다른 사람이 미세 튜닝한 사전 훈련된 모델을 사용해 영화 리뷰를 분류한다.

    ![표현 모델 활용](./image/04_representation_model_usage.png)

## Chapter 4-3 모델 선택 <a class="anchor" id="chapter4-3"></a>
1. GPT 패밀리와 같은 생성 모델이 텍스트 분류에 놀라운 성능을 내지만, 인코더 기반 모델도 특정 작업에서 강력한 성능을 가지며 일반적으로 크기가 훨씬 작다.

2. 작업에 맞는 모델을 선택하는 것은 일종의 예술에 가깝다.
    - 허깅 페이스 허브에 있는 수천 개의 사전 훈련된 모델을 모두 테스트 하는 것은 불가능에 가깝다.
    
3. 몇 개의 모델은 시작점으로 훌련한 선택이며 다양한 종류의 모델 성능을 가늠하는 기준으로 삼을 수 있다.
    - BERT, RoBERTa, DistilBERT, ALBERT,  DeBERTa  등이 있다.

4. 감성 분석을 위한 모델로 Twitter-RoBERTa-base를 선택한다.
    - 트위터에서 수집한 58억 개의 단어로 훈련된 모델로, 감성 분석과 같은 작업에 특히 적합하다.
    - 영화 리뷰에서 훈련된 모델은 아니지만 이 모델이 얼마나 일반화가 잘되는지 실펴보자.

5. 임베딩을 생성하는 모델을 선택할 때 우선 MTEB 리더보드를 찾아보는 것이 좋다.
    - MTEB는 다양한 다운스트림 작업에서 모델의 성능을 평가하는 벤치마크이다.
    - 모델이 텍스트 분류 작업에서 어떻게 수행되는지 확인할 수 있다.

6. 임베딩을 생성하는 모델로 sentence-transformers/all-mpnet-base-v2를 선택한다.
    - 이 모델은 1억 개 이상의 문장으로 훈련된 모델로, 텍스트 분류와 같은 다운스트림 작업에서 좋은 성능을 보인다.

## Chapter 4-4 작업에 특화된 모델 사용하기 <a class="anchor" id="chapter4-4"></a>

In [3]:
# Twitter-RoBERTa-base 모델 로드
from transformers import pipeline

# 허깅 페이스 모델 경로
model_path = " cardiffnlp/twitter-roberta-base-sentiment-latest"

# 파이프라인으로 모델 로드
from transformers import pipeline

# 허깅 페이스 모델 경로
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# 파이프라인으로 모델을 로드합니다.
pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    top_k=None,
    device="cuda:0"
)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


vocab.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cuda:0


1. 모델을 로드할 때 토크나이져로 로드되며 아래의 이미지와 같은 작업을 수행한다.
    - 입력 문장은 먼저 토크나이저를 통해 토큰으로 분할된 후 작업에 특화된 모델에 의해 처리된다.

        ![모델 로드](./image/04_model_load.png)

    - 훈련 데이터에 없는 단어를 만나더라도 토큰을 결합하여 표현을 생성할 수 있다.

        ![모델 로드](./image/04_model_load2.png)

In [5]:
# 테스트 세트로 모델을 실행한다.
import numpy as np
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset

# 추론 실행
y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])):
    scores = {item["label"]: item["score"] for item in output}
    negative_score = scores["negative"]
    positive_score = scores["positive"]
    assigment = np.argmax([negative_score, positive_score])
    y_pred.append(assigment)

Disabling tokenizer parallelism, we're using DataLoader multithreading already
100%|██████████| 1066/1066 [00:07<00:00, 151.85it/s]


In [6]:
from sklearn.metrics import classification_report

def evaluate_performance(y_true, y_pred):
    performance = classification_report(y_true, y_pred, target_names=["Negative Review", "Positive Review"])
    return performance

In [7]:
evaluate_performance(data["test"]["label"], y_pred)

'                 precision    recall  f1-score   support\n\nNegative Review       0.76      0.88      0.81       533\nPositive Review       0.86      0.72      0.78       533\n\n       accuracy                           0.80      1066\n      macro avg       0.81      0.80      0.80      1066\n   weighted avg       0.81      0.80      0.80      1066\n'

2. 분류 리포트에서 올바른 예측과 잘못된 예측을 구별하는 방법
    - 올바른 예측 대비 잘못된 예측, 올바른 클래스 대비 잘못된 클래스에 따라 네 개의 조합이 생성된다.
    - 혼동행렬이라고 부르며 모델의 성능을 시각적으로 표현하는 방법이다.

        ![혼동행렬](./image/04_confusion_matrix.png)

3. 혼동 행렬을 사용해 모델의 품질을 나타내는 여러 공식을 유도할 수 있다.
    - 정확도(Accuracy), 정밀도(Precision), 재현율(Recall), F1 점수 등이 있다.
    
        ![공식](./image/04_metrics.png)

4. 사전 훈련된 BERT 모델의 F1 점수는 0.8(f1-score 열에서 weighted avg 행)러 영화 리뷰에서 훈련하지 않은 것을 생각하면 높은점수이다.
    - 성능이 높은 모델을 얻기 위해 이 예제와 같이 다양한 시도를 해 볼 수 있다.
      

## Chapter 4-5 임베딩을 활용하여 분류 작업 수행하기 <a class="anchor" id="chapter4-5"></a>
1. 특정 작업에 맞는 사전 훈련된 모델이 없는경우, 범용 임베딩 모델을 사용할 수 있다.

### Chapter 4-5-1 지도 학습 분류 <a class="anchor" id="chapter4-5-1"></a>
1. 분류 작업에 표현모델을 바로 사용하는 것이 아니라 임베딩 모델을 사용해 특성을 생성한다.
    - 그런 다음 이 특성을 사용하여 선형 분류기와 같은 간단한 모델을 훈련할 수 있다.

        ![임베딩 활용](./image/04_embedding_usage.png)

2. 분류기를 사요하는 이점은 비용이 많이 드는 임베딩 모델의 미세튜닝을 피할 수 있다는 것이다.

3. 첫 번째 단계는 임베딩 모델을 사용해 텍스트 입력을 임베딩으로 변환하는 것이다.
    - 이 모델은 동결되며 훈련되지 않는다. 단지 입력 텍스트를 벡터로 변환하는 역할만 한다.

In [9]:
# sentence-transformers 라이브러리를 사용하여 텍스트 임베딩 모델을 로드한다.
from sentence_transformers import SentenceTransformer

# 모델을 로드한다.
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# 텍스트를 임베딩으로 변환한다.
train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)

loading configuration file config.json from cache at /home/redinblue/.cache/huggingface/hub/models--sentence-transformers--all-mpnet-base-v2/snapshots/e8c3b32edf5434bc2275fc9bab85f82640a19130/config.json
Model config MPNetConfig {
  "architectures": [
    "MPNetForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "mpnet",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "relative_attention_num_buckets": 32,
  "transformers_version": "4.57.3",
  "vocab_size": 30527
}

loading weights file model.safetensors from cache at /home/redinblue/.cache/huggingface/hub/models--sentence-transformers--all-mpnet-base-v2/snapshots/e8c3b32edf5434bc2275fc9bab85f82640a19130/model.safetensors
loading file vocab.txt from cac

Batches:   0%|          | 0/267 [00:00<?, ?it/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

In [10]:
train_embeddings.shape, test_embeddings.shape

((8530, 768), (1066, 768))

4. 두 번째 단계는 임베딩 분류기에 입력 특성으로 제공한다.
    - 이 분류기는 훈련 가능하며 로지스틱 회귀뿐만 아니라 분류 작업을 수행할 수 있다면 어떤 모델도 가능하다.

In [11]:
from sklearn.linear_model import LogisticRegression

# 훈련 세트의 임베딩으로 로지스틱 회귀 모델을 훈련한다.
clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, data["train"]["label"])

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [12]:
y_pred = clf.predict(test_embeddings)
evaluate_performance(data["test"]["label"], y_pred)

'                 precision    recall  f1-score   support\n\nNegative Review       0.85      0.86      0.85       533\nPositive Review       0.86      0.85      0.85       533\n\n       accuracy                           0.85      1066\n      macro avg       0.85      0.85      0.85      1066\n   weighted avg       0.85      0.85      0.85      1066\n'

5. 실행 결과를 정리하면 아래의 이미지와 같다.
    - 임베딩 모델이 텍스트를 벡터로 변환하는 역할을 하고, 분류기는 이 벡터를 사용하여 예측을 수행한다.

        ![임베딩 활용 결과](./image/04_embedding_usage_result.png)

### Chapter 4-5-2 데이터에 레이블이 없는 경우 <a class="anchor" id="chapter4-5-2"></a>
1. 제로샷 분류는 데이터에서 훈련되지 않았더라도 입력 데이터 레이블을 예측한다.
    - 제로샷 분류에서는 레이블을 가진 데이터가 없고 레이블 자체만 있다.
    - 제로샷 모델은 입력이 후보 레이블과 어떻게 관련 있는지 결정한다.

        ![제로샷 분류](./image/04_zero_shot_classification.png)


2. 임베딩으로 제로샷 분류를 수행하기 위해 레이블이 나타내야 하는 것을 기반으로 레이블 설명을 만든다.
    - 예를 들어 영화의 긍정 또는 부정 감정을 나타내는 레이블 설명 "A negative movie review"와 "A positive movie review"를 만든다.
    - 그 다음 임베딩 모델을 통해 임베딩을 생성한다.
    
        ![제로샷 분류 임베딩](./image/04_zero_shot_embedding.png)

In [13]:
# 레이블 임베딩을 만든다.
label_embeddings = model.encode(["A negative movie review", "A positive movie review"])

3. 문서 레이블을 할당하기 위해 코사인 유사도를 문서-레이블 쌍에 적용한다.
    - 코사인 유사도는 두 벡터 또는 임베딩 사이의 코사인 값이다.
    - 문서와 두 개의 후보 레이블의 유사도를 계산하여 가장 유사한 레이블을 문서에 할당한다.
    - 이미지료 표현하면 아래와 같다.

        ![제로샷 분류 유사도](./image/04_zero_shot_similarity.png)

In [14]:
# scikit-learn의 cosine_similarity 함수를 사용하여 문서-레이블 쌍에 코사인 유사도를 적용한다.
from sklearn.metrics.pairwise import cosine_similarity

# 각 문서와 가장 잘 맞는 레이블을 찾는다.
sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

In [15]:
evaluate_performance(data["test"]["label"], y_pred)

'                 precision    recall  f1-score   support\n\nNegative Review       0.83      0.76      0.79       533\nPositive Review       0.78      0.85      0.81       533\n\n       accuracy                           0.80      1066\n      macro avg       0.80      0.80      0.80      1066\n   weighted avg       0.80      0.80      0.80      1066\n'

4. 결과를 이미지로 표현하면 아래와 같다.
    - F1 점수는 0.78로 표현 모델을 사용한 분류보다 약간 낮지만, 제로샷 분류는 레이블이 없는 데이터에서도 작동한다는 점에서 여전히 인상적이다.

        ![제로샷 분류 결과](./image/04_zero_shot_result.png)

## Chapter 4-6 생성 모델로 텍스트 분류하기 <a class="anchor" id="chapter4-6"></a>
